# Stage 4: Adversarial Attacks

**Part A — Warm-up: attack the FR decision**
- FGSM and I-FGSM on cosine similarity
- Obfuscation (genuine pairs: push apart) and impersonation (imposter pairs: pull together)
- Goal: confirm the gradient pipeline works

**Part B — Main contribution: attack the explanation**
- Keep FR decision intact, change the saliency map to point at irrelevant regions
- Loss = λ_pred × prediction_stability_loss + λ_expl × explanation_divergence_loss
- Optimized with I-FGSM-style iterations inside an ε-ball

**Evaluation metrics per pair:**
| Metric | Want |
|---|---|
| Δ cosine similarity | Small (decision preserved) |
| Saliency map correlation | Low (explanation changed) |
| L∞ perturbation | ≤ ε (imperceptible) |

In [ ]:
!pip install timm -q

## ⚙️ CONFIG — edit only this cell

In [ ]:
import torch

# -----------------------------------------------------------------
# Paths
IMAGES_DIR   = "/kaggle/input/YOUR-DATASET-SLUG/imgs"
WEIGHTS_PATH = "/kaggle/input/YOUR-DATASET-SLUG/ElasticFaceArc.pth"
OUTPUT_DIR   = "/kaggle/working"

# -----------------------------------------------------------------
# Dataset
N_PAIRS     = 900
N_GENUINE_1 = 300
N_IMPOSTER  = 300
NUM_FEATURES = 512
THRESHOLD   = 0.30

# -----------------------------------------------------------------
# Test subset (pairs used for attack evaluation)
# Genuine test: first N pairs from each genuine block
# Imposter test: first N pairs from the imposter block
N_TEST_GENUINE  = 50   # pairs 0-49 and 600-649
N_TEST_IMPOSTER = 50   # pairs 300-349

# -----------------------------------------------------------------
# FGSM / I-FGSM attack parameters
EPSILON     = 0.03    # L-inf perturbation budget
ALPHA       = 0.005   # step size per I-FGSM iteration
STEPS_IFGSM = 20      # number of I-FGSM iterations

# -----------------------------------------------------------------
# Explanation attack parameters
STEPS_EXPL   = 30     # I-FGSM iterations for explanation attack
LAMBDA_PRED  = 1.0    # weight for prediction stability loss
LAMBDA_EXPL  = 2.0    # weight for explanation divergence loss
# Stability thresholds in the prediction loss
# (genuine pair: penalize if sim < SIM_MARGIN_GENUINE)
# (imposter pair: penalize if sim > SIM_MARGIN_IMPOSTER)
SIM_MARGIN_GENUINE  = 0.5
SIM_MARGIN_IMPOSTER = 0.2

# Saliency smoothing for visualization (not used inside attack loop)
GRAD_SMOOTH_SIGMA = 2.0

# -----------------------------------------------------------------
# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## Imports

In [ ]:
import os
import numpy as np
import cv2
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

## Backbone, Data & Model

In [ ]:
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)

def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class IBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None,
                 groups=1, base_width=64, dilation=1, use_se=False):
        super().__init__()
        self.bn1        = nn.BatchNorm2d(inplanes, eps=1e-05)
        self.conv1      = conv3x3(inplanes, planes)
        self.bn2        = nn.BatchNorm2d(planes, eps=1e-05)
        self.prelu      = nn.PReLU(planes)
        self.conv2      = conv3x3(planes, planes, stride)
        self.bn3        = nn.BatchNorm2d(planes, eps=1e-05)
        self.downsample = downsample
        self.stride     = stride

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out


class IResNet(nn.Module):
    fc_scale = 7 * 7
    def __init__(self, block, layers, dropout=0, num_features=512,
                 groups=1, width_per_group=64,
                 replace_stride_with_dilation=None):
        super().__init__()
        self.inplanes   = 64
        self.dilation   = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        self.groups     = groups
        self.base_width = width_per_group
        self.conv1      = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1        = nn.BatchNorm2d(64, eps=1e-05)
        self.prelu      = nn.PReLU(64)
        self.layer1     = self._make_layer(block, 64,  layers[0], stride=2)
        self.layer2     = self._make_layer(block, 128, layers[1], stride=2,
                                           dilate=replace_stride_with_dilation[0])
        self.layer3     = self._make_layer(block, 256, layers[2], stride=2,
                                           dilate=replace_stride_with_dilation[1])
        self.layer4     = self._make_layer(block, 512, layers[3], stride=2,
                                           dilate=replace_stride_with_dilation[2])
        self.bn2        = nn.BatchNorm2d(512 * block.expansion, eps=1e-05)
        self.dropout    = nn.Dropout(p=dropout, inplace=True)
        self.fc         = nn.Linear(512 * block.expansion * self.fc_scale, num_features)
        self.features   = nn.BatchNorm1d(num_features, eps=1e-05)
        nn.init.constant_(self.features.weight, 1.0)
        self.features.weight.requires_grad = False
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0, 0.1)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, num_blocks, stride=1, dilate=False):
        downsample    = None
        prev_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion, eps=1e-05),
            )
        layers = [block(self.inplanes, planes, stride, downsample,
                        self.groups, self.base_width, prev_dilation)]
        self.inplanes = planes * block.expansion
        for _ in range(1, num_blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups,
                                base_width=self.base_width, dilation=self.dilation))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.features(x)
        return x


def elastic_iresnet100(**kwargs):
    return IResNet(IBasicBlock, [3, 13, 30, 3], **kwargs)


# ---- data helpers ----
def preprocess(img):
    img = ((img / 255.0) - 0.5) / 0.5
    return torch.tensor(np.transpose(img, (2, 0, 1)).astype('float32'))

def load_image_raw(pair_idx, which):
    path = os.path.join(IMAGES_DIR, f"{pair_idx * 2 + which}.jpg")
    img  = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Missing: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def load_pair(pair_idx):
    return (preprocess(load_image_raw(pair_idx, 0)).to(DEVICE),
            preprocess(load_image_raw(pair_idx, 1)).to(DEVICE))

def is_genuine(pair_idx):
    return pair_idx < N_GENUINE_1 or pair_idx >= N_GENUINE_1 + N_IMPOSTER

def tensor_to_uint8(t):
    img = t.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip((img * 0.5 + 0.5) * 255, 0, 255).astype(np.uint8)


# ---- model ----
def load_model():
    m = elastic_iresnet100(num_features=NUM_FEATURES)
    state = torch.load(WEIGHTS_PATH, map_location=DEVICE)
    if 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state)
    return m.to(DEVICE).eval()

model = load_model()
print("Model ready.")

---
# Part A — Warm-up: Attacking the FR Decision

Obfuscation (genuine pairs): maximize cosine distance → the model thinks they are different.  
Impersonation (imposter pairs): minimize cosine distance → the model thinks they are the same.

## Loss Functions

In [ ]:
def obfuscation_loss(img_a, img_b):
    """Genuine pairs: minimize similarity (push embeddings apart)."""
    e_a = model(img_a.unsqueeze(0))
    e_b = model(img_b.unsqueeze(0).detach())
    return F.cosine_similarity(e_a, e_b, dim=1).mean()


def impersonation_loss(img_a, img_b):
    """Imposter pairs: maximize similarity (pull embeddings together)."""
    e_a = model(img_a.unsqueeze(0))
    e_b = model(img_b.unsqueeze(0).detach())
    return -F.cosine_similarity(e_a, e_b, dim=1).mean()

## FGSM

In [ ]:
def fgsm(img_a_orig, img_b, genuine=True, epsilon=EPSILON):
    """
    Single-step FGSM on image A.
    Returns adversarial image A (same shape, clamped to [-1, 1]).
    """
    model.eval()
    loss_fn = obfuscation_loss if genuine else impersonation_loss

    img_a_adv = img_a_orig.clone().detach().requires_grad_(True)
    loss = loss_fn(img_a_adv, img_b)
    loss.backward()

    perturbation = epsilon * img_a_adv.grad.sign()
    return torch.clamp(img_a_orig + perturbation, -1, 1).detach()

## I-FGSM

In [ ]:
def ifgsm(img_a_orig, img_b, genuine=True,
          epsilon=EPSILON, alpha=ALPHA, steps=STEPS_IFGSM):
    """
    Iterative FGSM on image A, projected back into the ε-ball.
    Returns adversarial image A.
    """
    model.eval()
    loss_fn   = obfuscation_loss if genuine else impersonation_loss
    img_a_adv = img_a_orig.clone().detach()

    for _ in range(steps):
        img_a_adv = img_a_adv.requires_grad_(True)
        loss      = loss_fn(img_a_adv, img_b)
        loss.backward()

        with torch.no_grad():
            img_a_adv = img_a_adv + alpha * img_a_adv.grad.sign()
            delta     = torch.clamp(img_a_adv - img_a_orig, -epsilon, epsilon)
            img_a_adv = torch.clamp(img_a_orig + delta, -1, 1)

    return img_a_adv.detach()

## Warm-up Evaluation

In [ ]:
# Build test index lists
test_genuine_idx  = list(range(N_TEST_GENUINE // 2)) + \
                    list(range(N_GENUINE_1 + N_IMPOSTER,
                               N_GENUINE_1 + N_IMPOSTER + N_TEST_GENUINE // 2))
test_imposter_idx = list(range(N_GENUINE_1, N_GENUINE_1 + N_TEST_IMPOSTER))


@torch.no_grad()
def get_sim(img_a, img_b):
    return F.cosine_similarity(model(img_a.unsqueeze(0)),
                               model(img_b.unsqueeze(0)), dim=1).item()


def eval_decision_attack(attack_fn, pair_indices, genuine):
    results = []
    for idx in tqdm(pair_indices, desc=f"{'Genuine' if genuine else 'Imposter'}"):
        img_a, img_b = load_pair(idx)
        sim_orig = get_sim(img_a, img_b)
        img_adv  = attack_fn(img_a, img_b, genuine=genuine)
        sim_adv  = get_sim(img_adv, img_b)
        # Decision flipped if:
        #   genuine pair  : sim dropped below threshold
        #   imposter pair : sim rose above threshold
        flipped = (sim_orig > THRESHOLD) != (sim_adv > THRESHOLD)
        l_inf   = (img_adv - img_a).abs().max().item()
        results.append(dict(pair_idx=idx, sim_orig=sim_orig,
                            sim_adv=sim_adv, flipped=flipped, l_inf=l_inf))
    return results


print("=== FGSM ===")
res_fgsm_gen = eval_decision_attack(fgsm, test_genuine_idx,  genuine=True)
res_fgsm_imp = eval_decision_attack(fgsm, test_imposter_idx, genuine=False)

print("\n=== I-FGSM ===")
res_ifgsm_gen = eval_decision_attack(ifgsm, test_genuine_idx,  genuine=True)
res_ifgsm_imp = eval_decision_attack(ifgsm, test_imposter_idx, genuine=False)

In [ ]:
def summarize(results, label):
    n         = len(results)
    flipped   = sum(r['flipped'] for r in results)
    mean_linf = np.mean([r['l_inf'] for r in results])
    mean_dsim = np.mean([abs(r['sim_adv'] - r['sim_orig']) for r in results])
    print(f"{label:30s}  flip_rate={flipped/n:.2f}  "
          f"mean_Δsim={mean_dsim:.4f}  mean_L∞={mean_linf:.4f}")

summarize(res_fgsm_gen,  "FGSM    genuine")
summarize(res_fgsm_imp,  "FGSM    imposter")
summarize(res_ifgsm_gen, "I-FGSM  genuine")
summarize(res_ifgsm_imp, "I-FGSM  imposter")

## Visualize a Warm-up Attack

In [ ]:
def show_decision_attack(pair_idx, genuine=True):
    img_a, img_b = load_pair(pair_idx)
    img_adv      = ifgsm(img_a, img_b, genuine=genuine)
    perturbation = (img_adv - img_a).abs() * 10   # amplify for visibility

    sim_orig = get_sim(img_a, img_b)
    sim_adv  = get_sim(img_adv, img_b)
    label    = 'Genuine' if genuine else 'Imposter'

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, img, title in zip(axes,
                               [tensor_to_uint8(img_a),
                                tensor_to_uint8(img_adv),
                                tensor_to_uint8(perturbation.clamp(0,1))],
                               [f'Original  sim={sim_orig:.3f}',
                                f'Adversarial  sim={sim_adv:.3f}',
                                'Perturbation ×10']):
        ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis('off')
    fig.suptitle(f"I-FGSM on {label} Pair {pair_idx}", fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'ifgsm_{label.lower()}_pair{pair_idx}.png'), dpi=120)
    plt.show()

show_decision_attack(0,   genuine=True)
show_decision_attack(300, genuine=False)

---
# Part B — Main Contribution: Attacking the Explanation

The attack perturbs image A to change its saliency map while keeping the FR decision stable.

**Differentiable saliency** is used inside the optimization loop: squared gradient of cosine similarity w.r.t. the input image. This requires second-order gradients (`create_graph=True`), which PyTorch supports.

## Differentiable Saliency (used inside attack loop)

In [ ]:
def differentiable_saliency(img_a, img_b):
    """
    Squared gradient saliency: (∂sim/∂img_a)²  averaged over channels.
    Returns (H, W) tensor still attached to the computational graph
    (create_graph=True) so the attack loss can be backpropagated through it.
    img_a must already have requires_grad=True.
    """
    e_a = model(img_a.unsqueeze(0))
    e_b = model(img_b.unsqueeze(0).detach())
    sim = F.cosine_similarity(e_a, e_b, dim=1).mean()
    # create_graph=True keeps the graph so backward() can differentiate through this
    grad = torch.autograd.grad(sim, img_a, create_graph=True)[0]  # (3, H, W)
    return grad.pow(2).mean(dim=0)                                  # (H, W)


def saliency_for_display(img_a, img_b, sigma=GRAD_SMOOTH_SIGMA):
    """Non-differentiable version for visualization only — returns numpy in [0,1]."""
    model.eval()
    img_a_var = img_a.clone().requires_grad_(True)
    sal = differentiable_saliency(img_a_var, img_b).detach().cpu().numpy()
    if sigma > 0:
        sal = gaussian_filter(sal, sigma=sigma)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    return sal

## Explanation Attack

In [ ]:
def explanation_attack(img_a_orig, img_b, genuine=True,
                       epsilon=EPSILON, alpha=ALPHA, steps=STEPS_EXPL,
                       lambda_pred=LAMBDA_PRED, lambda_expl=LAMBDA_EXPL):
    """
    I-FGSM-style attack targeting the saliency map while preserving the FR decision.

    Loss = lambda_pred * pred_stability_loss + lambda_expl * explanation_divergence_loss

    pred_stability_loss:
      genuine  : relu(SIM_MARGIN_GENUINE  - sim)  — penalize if sim drops too much
      imposter : relu(sim - SIM_MARGIN_IMPOSTER)  — penalize if sim rises too much

    explanation_divergence_loss:
      -MSE(attacked_saliency, original_saliency)  — maximize map distance

    Returns adversarial image A.
    """
    model.eval()

    # Compute and freeze the original saliency map
    img_a_ref = img_a_orig.clone().detach().requires_grad_(True)
    with torch.enable_grad():
        original_map = differentiable_saliency(img_a_ref, img_b).detach()

    img_a_adv = img_a_orig.clone().detach()

    for step in range(steps):
        img_a_adv = img_a_adv.requires_grad_(True)

        # ── prediction stability ──────────────────────────────────────
        e_a = model(img_a_adv.unsqueeze(0))
        e_b = model(img_b.unsqueeze(0).detach())
        sim = F.cosine_similarity(e_a, e_b, dim=1).mean()

        if genuine:
            pred_loss = F.relu(SIM_MARGIN_GENUINE - sim)
        else:
            pred_loss = F.relu(sim - SIM_MARGIN_IMPOSTER)

        # ── explanation divergence (second-order gradient) ────────────
        attacked_map = differentiable_saliency(img_a_adv, img_b)
        expl_loss    = -F.mse_loss(attacked_map, original_map)

        total_loss = lambda_pred * pred_loss + lambda_expl * expl_loss
        total_loss.backward()

        with torch.no_grad():
            img_a_adv = img_a_adv + alpha * img_a_adv.grad.sign()
            delta     = torch.clamp(img_a_adv - img_a_orig, -epsilon, epsilon)
            img_a_adv = torch.clamp(img_a_orig + delta, -1, 1)

    return img_a_adv.detach()

## Full Evaluation

In [ ]:
def evaluate_pair(pair_idx, img_a_orig, img_a_adv, img_b):
    """Compute all three metrics for one pair."""
    with torch.no_grad():
        sim_orig = F.cosine_similarity(
            model(img_a_orig.unsqueeze(0)), model(img_b.unsqueeze(0)), dim=1).item()
        sim_adv  = F.cosine_similarity(
            model(img_a_adv.unsqueeze(0)),  model(img_b.unsqueeze(0)), dim=1).item()

    # Saliency map correlation (Pearson r)
    sal_orig = saliency_for_display(img_a_orig, img_b, sigma=0).flatten()
    sal_adv  = saliency_for_display(img_a_adv,  img_b, sigma=0).flatten()
    corr, _  = pearsonr(sal_orig, sal_adv)

    # Perturbation
    delta = (img_a_adv - img_a_orig).abs()
    l_inf = delta.max().item()
    l2    = delta.pow(2).sum().sqrt().item()

    return dict(
        pair_idx          = pair_idx,
        genuine           = is_genuine(pair_idx),
        sim_orig          = sim_orig,
        sim_adv           = sim_adv,
        decision_preserved= (sim_orig > THRESHOLD) == (sim_adv > THRESHOLD),
        map_correlation   = corr,
        l_inf             = l_inf,
        l2                = l2,
    )


expl_results = []
all_test = test_genuine_idx + test_imposter_idx

for idx in tqdm(all_test, desc="Explanation attacks"):
    img_a, img_b = load_pair(idx)
    img_adv      = explanation_attack(img_a, img_b, genuine=is_genuine(idx))
    metrics      = evaluate_pair(idx, img_a, img_adv, img_b)
    expl_results.append(metrics)

print(f"Done — {len(expl_results)} pairs evaluated.")

## Results Summary

In [ ]:
gen_res = [r for r in expl_results if     r['genuine']]
imp_res = [r for r in expl_results if not r['genuine']]

def print_summary(results, label):
    n         = len(results)
    preserved = sum(r['decision_preserved'] for r in results) / n
    corr      = np.mean([r['map_correlation'] for r in results])
    l_inf     = np.mean([r['l_inf'] for r in results])
    l2        = np.mean([r['l2'] for r in results])
    print(f"{label} (n={n})")
    print(f"  Decision preserved : {preserved:.2%}")
    print(f"  Map correlation    : {corr:.4f}  (lower = more change)")
    print(f"  Mean L∞            : {l_inf:.4f}  (budget = {EPSILON})")
    print(f"  Mean L2            : {l2:.4f}")

print_summary(gen_res, "Genuine pairs")
print()
print_summary(imp_res, "Imposter pairs")

## Visualize: Before vs After Explanation Attack

In [ ]:
def overlay(img_tensor, sal_np, alpha=0.5):
    raw = tensor_to_uint8(img_tensor)
    hm  = cv2.applyColorMap((sal_np * 255).astype(np.uint8), cv2.COLORMAP_JET)
    hm  = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
    return np.clip(alpha * hm + (1 - alpha) * raw, 0, 255).astype(np.uint8)


def visualize_explanation_attack(pair_idx):
    img_a, img_b = load_pair(pair_idx)
    genuine      = is_genuine(pair_idx)
    img_adv      = explanation_attack(img_a, img_b, genuine=genuine)

    sal_orig = saliency_for_display(img_a,   img_b)
    sal_adv  = saliency_for_display(img_adv, img_b)
    corr, _  = pearsonr(sal_orig.flatten(), sal_adv.flatten())

    sim_orig = get_sim(img_a, img_b)
    sim_adv  = get_sim(img_adv, img_b)
    label    = 'Genuine' if genuine else 'Imposter'

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    axes[0, 0].imshow(tensor_to_uint8(img_a))
    axes[0, 0].set_title(f'Original A  sim={sim_orig:.3f}')

    axes[0, 1].imshow(overlay(img_a, sal_orig))
    axes[0, 1].set_title('Original saliency')

    axes[0, 2].imshow(tensor_to_uint8(img_b))
    axes[0, 2].set_title('Image B (unchanged)')

    axes[1, 0].imshow(tensor_to_uint8(img_adv))
    axes[1, 0].set_title(f'Adversarial A  sim={sim_adv:.3f}')

    axes[1, 1].imshow(overlay(img_adv, sal_adv))
    axes[1, 1].set_title(f'Attacked saliency  (corr={corr:.3f})')

    pert = ((img_adv - img_a).abs() * 10).clamp(0, 1)
    axes[1, 2].imshow(tensor_to_uint8(pert))
    axes[1, 2].set_title(f'Perturbation ×10  L∞={pert.max()/10:.4f}')

    for ax in axes.flatten(): ax.axis('off')
    fig.suptitle(f"Explanation Attack — {label} Pair {pair_idx}", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,
                             f'expl_attack_{label.lower()}_pair{pair_idx}.png'), dpi=120)
    plt.show()


# Show two genuine and one imposter example
visualize_explanation_attack(test_genuine_idx[0])
visualize_explanation_attack(test_genuine_idx[1])
visualize_explanation_attack(test_imposter_idx[0])

## Metric Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Similarity change
ax = axes[0]
for res, label, color in [(gen_res, 'Genuine', 'green'), (imp_res, 'Imposter', 'red')]:
    sims = [r['sim_adv'] - r['sim_orig'] for r in res]
    ax.hist(sims, bins=20, alpha=0.6, color=color, label=label)
ax.axvline(0, color='black', lw=1, linestyle='--')
ax.set_title('Δ Cosine Similarity (adv − orig)')
ax.set_xlabel('Δ sim'); ax.legend()

# 2. Saliency map correlation
ax = axes[1]
for res, label, color in [(gen_res, 'Genuine', 'green'), (imp_res, 'Imposter', 'red')]:
    corrs = [r['map_correlation'] for r in res]
    ax.hist(corrs, bins=20, alpha=0.6, color=color, label=label)
ax.set_title('Saliency Map Correlation (lower = more disruption)')
ax.set_xlabel('Pearson r'); ax.legend()

# 3. L-inf perturbation
ax = axes[2]
for res, label, color in [(gen_res, 'Genuine', 'green'), (imp_res, 'Imposter', 'red')]:
    linfs = [r['l_inf'] for r in res]
    ax.hist(linfs, bins=20, alpha=0.6, color=color, label=label)
ax.axvline(EPSILON, color='black', lw=1.5, linestyle='--', label=f'ε = {EPSILON}')
ax.set_title('L∞ Perturbation')
ax.set_xlabel('L∞'); ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metric_distributions.png'), dpi=150)
plt.show()